# Unitree G1 — Flat Terrain Training
Train con robot đi trên mặt phẳng bằng MuJoCo Playground + Brax PPO trên Kaggle Dual T4.

In [15]:
# ============================================================
# Cell 1: Kiểm tra GPU
# ============================================================
import subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"])

0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], returncode=0)

In [3]:
# ============================================================
# Cell 2: Clone repo & cd vào project
# ============================================================
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")

if not PROJECT_DIR.exists():
    subprocess.check_call([
        "git", "clone",
        "--branch", "M2",
        "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git",
        str(PROJECT_DIR)
    ])
    print("[✓] Cloned repo")
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
    print("[✓] Repo up to date")

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

Already up to date.
[✓] Repo up to date
Working dir: /kaggle/working/Bipedal-Locomotion


In [2]:
# ============================================================
# Cell 3: Cài đặt dependencies
# ============================================================
import sys, subprocess

pkgs = [
    "brax",
    "flax",
    "optax",
    "orbax-checkpoint",
    "ml_collections",
    "pyyaml",
    "imageio",
    "imageio-ffmpeg",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

# MuJoCo Playground (v0.2.0 có G1JoystickFlatTerrain)
try:
    import mujoco_playground
    print("[✓] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"
    ])
    print("[✓] mujoco_playground installed")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[✓] mujoco_playground already installed


In [3]:
# ============================================================
# Cell 4: Kiểm tra JAX & import
# ============================================================
import os

# Phải set TRƯỚC KHI import mujoco để EGL được dùng cho rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import jax
devices = jax.devices()
print(f"[✓] JAX {jax.__version__} — {len(devices)} device(s)")
for d in devices:
    print(f"    {d}")

import mujoco_playground
print("[✓] mujoco_playground imported OK")
print(f"[✓] MUJOCO_GL={os.environ.get('MUJOCO_GL','(not set)')}")


[✓] JAX 0.11.1 — 2 device(s)
    cuda:0
    cuda:1
[✓] mujoco_playground imported OK
[✓] MUJOCO_GL=egl


In [ ]:
import importlib.util
import os
import sys
import yaml

PROJECT_DIR = "/kaggle/working/Bipedal-Locomotion"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

spec = importlib.util.spec_from_file_location("train_flat", f"{PROJECT_DIR}/scripts/train_flat.py")
train_flat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_flat)

with open("configs/ppo_flat.yaml") as f:
    cfg = yaml.safe_load(f)

meta = train_flat.train(cfg, output_dir=cfg["experiment"].get("output_dir", "results/flat"))
print("\n=== DONE ===")
print("Checkpoint:", meta["checkpoint_dir"])
print("Total time:", meta["total_wall_time_s"], "s")

[+] JAX devices (2): ['cuda:0', 'cuda:1']
[+] Output: /kaggle/working/Bipedal-Locomotion/results/flat/g1_flat_20261125_043013
[+] obs_size={'privileged_state': (216,), 'state': (103,)}  action_size=29
[+] Starting PPO training...
  step=         0  elapsed=   274s  sps=         0  eval_reward= -3.508  eval_len=  54.8
  step= 2,785,280  elapsed=   716s  sps=     3,891  eval_reward= -2.048  eval_len=  47.3
  step= 5,570,560  elapsed=   971s  sps=     5,735  eval_reward= -1.807  eval_len=  49.6
  step= 8,355,840  elapsed=  1136s  sps=     7,358  eval_reward= -1.655  eval_len=  55.1
  step=11,141,120  elapsed=  1300s  sps=     8,569  eval_reward= -1.636  eval_len=  58.6
  step=13,926,400  elapsed=  1464s  sps=     9,510  eval_reward= -1.643  eval_len=  60.6
  step=16,711,680  elapsed=  1629s  sps=    10,260  eval_reward= -1.582  eval_len=  63.7
  step=19,496,960  elapsed=  1793s  sps=    10,873  eval_reward= -1.520  eval_len=  67.2
  step=22,282,240  elapsed=  1957s  sps=    11,383  eval_r

In [12]:
# ============================================================
# Cell 6 (tuỳ chọn): Xem metrics sau khi train xong
# ============================================================
import glob, json

run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
if not run_dirs:
    print("Chưa có kết quả train")
else:
    latest = run_dirs[-1]
    rows = []
    with open(f"{latest}/metrics.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))

    print(f"Run: {latest}")
    print(f"{'step':>12}  {'eval_reward':>12}  {'eval_len':>10}  {'wall_time_s':>12}")
    for r in rows:
        print(
            f"{r.get('step', 0):>12,}  "
            f"{r.get('eval/episode_reward', float('nan')):>12.3f}  "
            f"{r.get('eval/avg_episode_length', float('nan')):>10.1f}  "
            f"{r.get('wall_time_s', 0):>12.0f}"
        )

Run: results/flat/g1_flat_20261125_043013
        step   eval_reward    eval_len   wall_time_s
           0        -3.508        54.8           274
   2,785,280        -2.048        47.3           716
   5,570,560        -1.807        49.6           971
   8,355,840        -1.655        55.1          1136
  11,141,120        -1.636        58.6          1300
  13,926,400        -1.643        60.6          1464
  16,711,680        -1.582        63.7          1629
  19,496,960        -1.520        67.2          1793
  22,282,240        -1.420        74.9          1957
  25,067,520        -1.436        80.3          2122
  27,852,800        -1.311        81.3          2286
  30,638,080        -0.760       114.0          2450
  33,423,360        -0.517       148.7          2615
  36,208,640         0.463       221.4          2779
  38,993,920         3.683       412.1          2944
  41,779,200         5.265       473.9          3108
  44,564,480         6.534       517.5          3272
  47

In [5]:
# ============================================================
# Cell 7: Render video demo (chạy trong subprocess riêng)
# Tránh OOM vì không dùng chung VRAM với kernel training
# ============================================================
import glob
import subprocess
import sys
from base64 import b64encode
from pathlib import Path

from IPython.display import HTML

# Tìm checkpoint mới nhất
run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
assert run_dirs, "Chưa có checkpoint!"
latest_run = Path(run_dirs[-1]).resolve()
ckpt_root = latest_run / "checkpoints"
step_dirs = sorted(
    [d for d in ckpt_root.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
ckpt_path = str((step_dirs[-1] if step_dirs else ckpt_root).resolve())
video_path = "/kaggle/working/g1_flat_demo.mp4"

print(f"[+] Checkpoint: {ckpt_path}")
print("[+] Chạy render_demo.py trong subprocess...")

result = subprocess.run(
    [
        sys.executable, "scripts/render_demo.py",
        "--checkpoint", ckpt_path,
        "--out", video_path,
        "--steps", "300",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)
    raise RuntimeError(f"render_demo.py failed (code {result.returncode})")

print(f"[✓] Video: {video_path}")
mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(
    f'<video width="480" height="360" controls autoplay loop>'
    f'<source src="{data_url}" type="video/mp4"></video>'
))


[+] Checkpoint: /kaggle/working/Bipedal-Locomotion/results/flat/g1_flat_20261125_043013/checkpoints/000052920320
[+] Chạy render_demo.py trong subprocess...

=== STDERR ===
/usr/bin/python3: can't open file '/kaggle/working/Bipedal-Locomotion/scripts/render_demo.py': [Errno 2] No such file or directory



RuntimeError: render_demo.py failed (code 2)